# Agent loop from scratch

An agent loop is a `while` loop. The model either calls a tool or it is done.

1. Send the messages and the tool list.
2. If the model asked for tools, run them and append the results.
3. Call the model again.
4. Stop when it replies with text instead of another tool call.


In [ ]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console

load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
client = OpenAI(base_url=GEMINI_BASE_URL, api_key=os.getenv("GOOGLE_API_KEY"))
MODEL = "gemini-3.5-flash"
console = Console()


## Tools

The model cannot tick a list by itself. These two Python functions are the only actions it can take: write the plan, then mark one step done.


In [ ]:
steps = []
done = []


def show_plan() -> str:
    lines = []
    for number, step in enumerate(steps, start=1):
        if done[number - 1]:
            lines.append(f"{number}. [green][strike]{step}[/strike][/green]")
        else:
            lines.append(f"{number}. {step}")
    report = "\n".join(lines)
    console.print(report)
    return report


def set_plan(items: list[str]) -> str:
    """Replace the plan with a new list of steps."""
    steps.clear()
    done.clear()
    steps.extend(items)
    done.extend([False] * len(items))
    return show_plan()


def complete_step(number: int, note: str) -> str:
    """Mark step `number` done. Numbers start at 1."""
    if not 1 <= number <= len(steps):
        return f"There is no step {number}."
    done[number - 1] = True
    console.print(f"[bold]Step {number}:[/bold] {note}")
    return show_plan()


## Describe the tools in JSON

The model only sees the name, description, and parameters. It never sees the Python.


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "set_plan",
            "description": "Set the plan to this ordered list of steps, replacing any earlier plan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "The steps, in the order they should be done",
                    }
                },
                "required": ["items"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "complete_step",
            "description": "Mark one plan step complete and record a short note about the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "number": {
                        "type": "integer",
                        "description": "The step number, starting at 1",
                    },
                    "note": {
                        "type": "string",
                        "description": "What was decided or calculated for this step",
                    },
                },
                "required": ["number", "note"],
                "additionalProperties": False,
            },
        },
    },
]

available = {"set_plan": set_plan, "complete_step": complete_step}


## The loop

`finish_reason == "tool_calls"` means the model is not finished. Run every tool it asked for, give the results back, and ask again.


In [ ]:
def run_tools(tool_calls):
    results = []
    for call in tool_calls:
        name = call.function.name
        arguments = json.loads(call.function.arguments)
        tool = available.get(name)
        output = tool(**arguments) if tool else f"Unknown tool: {name}"
        results.append(
            {"role": "tool", "content": json.dumps(output), "tool_call_id": call.id}
        )
    return results


def agent_loop(messages):
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        assistant = response.choices[0].message
        messages.append(assistant)
        messages.extend(run_tools(assistant.tool_calls))
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    console.print(response.choices[0].message.content)


## Ask it something that needs several steps

The model should set a plan, complete each step, then answer. It should not ask a follow-up question.


In [ ]:
system = """
You solve the user's problem with the plan tools.
First call set_plan with the steps. Then call complete_step for each step, in order, with the result of that step in the note.
If a number is missing, estimate it in its own step and say that it is an estimate.
After every step is complete, reply with the answer in Rich markup and no code blocks.
Do not ask the user anything.
"""

question = """
A team of 4 people wants a 45 minute introduction to agent loops.
They have never called a tool from a model before.
Plan the session, then give a timed agenda.
"""

messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": question},
]

steps.clear()
done.clear()
agent_loop(messages)
